In [1]:
import torch
from pathlib import Path
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import xml.etree.ElementTree as ET
from torch.utils.data import DataLoader
import torchvision.transforms as T
device = torch.device('mps')
from torch.utils.data import Dataset, DataLoader, random_split


In [2]:
folder_with_images = Path(r"/Users/igor.varha/CCP_workdir/skull_workdir/head_image_top")
folder_with_labels = Path(r"/Users/igor.varha/CCP_workdir/skull_workdir/head_label_top")


In [3]:
import os
import json
from PIL import Image
import torchvision.transforms as T

class AnchorPointDataset(Dataset):
    def __init__(self, json_folder, transform=None):
        """
        Args:
            json_folder (str): Path to the folder containing LabelMe JSON files.
            transform (callable, optional): Transform to be applied to the images.
        """
        self.json_folder = json_folder
        self.transform = transform or T.ToTensor()
        self.data = []

        # Parse all JSON files in the folder
        for json_file in os.listdir(json_folder):
            if json_file.endswith(".json"):
                with open(os.path.join(json_folder, json_file), 'r') as f:
                    data = json.load(f)
                    image_path = os.path.join(json_folder, data["imagePath"])
                    points = []
                    for shape in data["shapes"]:
                        points.extend(shape["points"])  # Add all points in the shape
                    points = [point for xy in points for point in xy]  # Flatten the list of points
                    self.data.append({"image_path": image_path, "points": points})

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        points = torch.tensor(sample["points"], dtype=torch.float32)  # List of points (x1, y1, x2, y2)

        if self.transform:
            image = self.transform(image)
        points = points.view(-1, 2)  # Reshape to (num_points, 2)
        return image, points


mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # Example for ImageNet
train_transforms = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=mean, std=std)
])
# Create the dataset
dataset = AnchorPointDataset(folder_with_images, transform=train_transforms)

# Split into train and test sets
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# DataLoaders

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
print("data 0", dataset[0])

data 0 (tensor([[[-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         ...,
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179]],

        [[-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         ...,
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357]],

        [[-1.8044, -1.8044, -1.8044,  ..., -1.8044, -1.8044, -1.8044],
         [-1.8044, -1.8044, -1.8044, 

In [4]:
import torch
import torch.nn as nn
from torchvision.models import vgg16

class AnchorPointPredictorWithVGG(nn.Module):
    def __init__(self, num_points=2):
        super().__init__()
        # Load pretrained VGG16
        self.backbone = vgg16(pretrained=True).features  # Extract feature layers

        # Regression head for anchor points
        self.regressor = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # Global average pooling
            nn.Flatten(),
            nn.Linear(512, 256),  # VGG outputs 512 features
            nn.Dropout(0.15),  # Dropout with 30% probability
            nn.ReLU(),
            nn.Dropout(0.15),  # Dropout with 30% probability
            nn.Linear(256, num_points * 2),  # Predict (x1, y1, x2, y2)
            nn.Sigmoid()  # Constrain outputs to [0, 1]
        )

    def forward(self, x):
        features = self.backbone(x)  # Extract features
        points = self.regressor(features)  # Predict anchor points
        return points.view(x.size(0), -1, 2)  # Reshape to (batch_size, num_points, 2)

# Initialize model
model = AnchorPointPredictorWithVGG(num_points=2)

/Users/igor.varha/miniconda3/envs/mouseSkullPipelines/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/igor.varha/miniconda3/envs/mouseSkullPipelines/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, random_split

# Define Loss Function
criterion = nn.L1Loss(reduction='sum')

# Loss function and optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Learning rate scheduler
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max=10,eta_min=0.000000001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)
# Device
device = torch.device("mps")
model.to(device)

# Training Parameters
num_epochs = 1
log_interval = 10  # Log every 10 batches
# Training and validation loop
num_epochs = 100
for epoch in range(1, num_epochs + 1):
    # Training phase
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images = images.to(device)
        targets = targets.view(targets.size(0), -1, 2).to(device)
        targets = normalize_targets(targets, images.shape[-2:])
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, targets)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            targets = targets.view(targets.size(0), -1, 2).to(device)

            outputs = model(images)
            outputs = denormalize_outputs(outputs, images.shape[-2:])
            loss = criterion(outputs, targets)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)
    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, "
          f"Validation Loss: {avg_val_loss:.4f}, LR: {scheduler.get_last_lr()}")

/Users/igor.varha/miniconda3/envs/mouseSkullPipelines/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/100], Train Loss: 0.8730, Validation Loss: 535.3000, LR: [0.001]
Epoch [2/100], Train Loss: 0.8421, Validation Loss: 535.3000, LR: [0.001]
Epoch [3/100], Train Loss: 0.8421, Validation Loss: 535.3000, LR: [0.001]
Epoch [4/100], Train Loss: 0.8421, Validation Loss: 535.3000, LR: [0.001]
Epoch [5/100], Train Loss: 0.8421, Validation Loss: 535.3000, LR: [0.001]


KeyboardInterrupt: 

In [123]:
import torch

# Switch model to evaluation mode
model.eval()

# Perform testing
with torch.no_grad():
    for batch_idx, (images, targets) in enumerate(test_loader, start=1):
        # Move data to device
        images = images.to(device)
        targets = targets.view(targets.size(0), -1, 2).to(device)  # Reshape to (batch_size, num_points, 2)





        # Predict anchor points
        predictions = model(images)  # Shape: (batch_size, num_points, 2)
        predictions = denormalize_outputs(predictions, images.shape[-2:])
        # Calculate differences between predictions and targets
        differences = predictions - targets  # Element-wise difference

        # Log results
        print(f"Batch [{batch_idx}/{len(test_loader)}]:")
        for i in range(images.size(0)):  # Iterate over batch
            print(f"  Image {i + 1}:")
            print(f"    Predicted Points: {predictions[i].cpu().numpy()}")
            print(f"    Target Points:    {targets[i].cpu().numpy()}")
            print(f"    Differences:      {differences[i].cpu().numpy()}")

        # Optional: Break after one batch to verify output


Batch [1/10]:
  Image 1:
    Predicted Points: [[376.11475 265.23828]
 [359.97153  97.03224]]
    Target Points:    [[417.5 249.5]
 [435.5  93.5]]
    Differences:      [[-41.385254   15.738281 ]
 [-75.52847     3.5322418]]
Batch [2/10]:
  Image 1:
    Predicted Points: [[408.262   305.2321 ]
 [391.21075 118.77733]]
    Target Points:    [[430.5 245. ]
 [425.5  90. ]]
    Differences:      [[-22.238007  60.232086]
 [-34.289246  28.777328]]
Batch [3/10]:
  Image 1:
    Predicted Points: [[454.053   279.11026]
 [434.88306 104.59341]]
    Target Points:    [[433.5 305.5]
 [442.5 124.5]]
    Differences:      [[ 20.553009  -26.38974  ]
 [ -7.6169434 -19.906593 ]]
Batch [4/10]:
  Image 1:
    Predicted Points: [[409.10138 280.04666]
 [391.77057 105.08409]]
    Target Points:    [[433.5 250.5]
 [441.5  92.5]]
    Differences:      [[-24.39862   29.546661]
 [-49.72943   12.584091]]
Batch [5/10]:
  Image 1:
    Predicted Points: [[404.67517 301.73935]
 [387.75452 116.86057]]
    Target Points:

In [6]:
def normalize_targets(targets, image_size):
    """
    Normalize targets to [0, 1] range based on image dimensions.

    Args:
        targets (Tensor): Tensor of shape (batch_size, num_points, 2).
        image_size (tuple): (width, height) of the image.

    Returns:
        Tensor: Normalized targets.
    """
    width, height = image_size
    targets[:, :, 0] /= width  # Normalize x-coordinates
    targets[:, :, 1] /= height  # Normalize y-coordinates
    return targets

def denormalize_outputs(outputs, image_size):
    """
    Denormalize predicted outputs from [0, 1] range to pixel coordinates.

    Args:
        outputs (Tensor): Tensor of shape (batch_size, num_points, 2).
        image_size (tuple): (width, height) of the image.

    Returns:
        Tensor: Denormalized outputs.
    """
    width, height = image_size
    outputs[:, :, 0] *= width  # Denormalize x-coordinates
    outputs[:, :, 1] *= height  # Denormalize y-coordinates
    return outputs

In [125]:
# iterate over images and copy one without label to the aut_label_dir
import os
for image_file in os.listdir(folder_with_images):
    if image_file.endswith(".xml"):
        continue
    if (folder_with_labels/Path(image_file).with_suffix(".xml")).exists():
        print(f"Skipping {image_file}", (folder_with_labels/Path(image_file).with_suffix(".xml")))
        continue
    # copy image file to aut_label_dir
    os.system(f"cp {folder_with_images/image_file} {folder_with_labels}")

def predict_pts( model, image_file, device):
    image = Image.open(folder_with_images/image_file).convert("RGB")
    image = train_transforms(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        prediction = model(image)
    return denormalize_outputs(prediction, image.shape[-2:]),image.shape[-2:]

def save_points_to_json(points, image_name, output_path, height,width,label="lab1"):
    """
    Save predicted points to JSON in the specified format.

    Args:
        points (list of tuples): List of anchor points [(x1, y1), (x2, y2), ...].
        image_name (str): Name of the image file.
        output_path (str): Path to save the JSON file.
        label (str): Label for the shape. Default is "lab1".

    Returns:
        None
    """
    data = {
        "version": "5.5.0",
        "flags": {},
        "shapes": [
            {
                "label": label,
                "points": points,
                "group_id": None,
                "description": "",
                "shape_type": "line",
                "flags": {},
                "mask": None
            }
        ],
        "imagePath": image_name,
        "imageData": None,
        "imageHeight": height,
        "imageWidth": width
    }

    # Save to JSON file
    with open(output_path, 'w') as json_file:
        json.dump(data, json_file, indent=2)

# iterate over images and make predictions and save to json file predictions

for image_file in folder_with_labels.iterdir():
    if image_file.suffix == ".json":
        continue
    if image_file.is_dir():
        continue
    if image_file.suffix not in [".png", ".jpg", ".jpeg"]:
        continue
    pts,shape = predict_pts(model, image_file, device)
    print(pts.cpu().numpy().tolist()[0])

    save_points_to_json(pts.cpu().numpy().tolist()[0], image_file.name, folder_with_labels/Path(image_file).with_suffix(".json"),height=shape[0],width=shape[1])




[[397.9471130371094, 281.1935729980469], [381.0595703125, 105.79084014892578]]
[[440.2893981933594, 312.4212646484375], [421.9349670410156, 122.66509246826172]]
[[384.96063232421875, 289.6433410644531], [368.7223815917969, 110.3891372680664]]
[[401.5729064941406, 293.3728332519531], [384.65167236328125, 112.34681701660156]]
[[415.1347961425781, 293.3096618652344], [397.6737365722656, 112.252685546875]]
[[387.1985168457031, 292.6953430175781], [370.92230224609375, 111.94930267333984]]
[[423.1978759765625, 294.9747009277344], [405.4631652832031, 113.18310546875]]
[[370.44842529296875, 276.5447082519531], [354.69635009765625, 103.33191680908203]]
[[455.85125732421875, 310.6328125], [436.8888854980469, 121.7591781616211]]
[[384.1286926269531, 316.8597106933594], [368.1382141113281, 125.05972290039062]]
[[422.5323791503906, 295.5890197753906], [404.8400573730469, 113.46317291259766]]
[[431.74652099609375, 292.4443359375], [413.6489562988281, 111.78887939453125]]
[[382.1968688964844, 288.309

In [93]:
# find images with no labels and label them with trained model

def predict_image(model, image, device, threshold, num_classes):
    """
    Predicts bounding boxes and labels for an image using a trained model.

    Args:
        model: Trained object detection model.
        image: PIL Image to be predicted.
        device: Device to perform inference (e.g., "cpu" or "cuda").
        threshold: Confidence threshold for filtering predictions.
        num_classes: Total number of object classes.

    Returns:
        Tuple of filtered bounding boxes and labels.
    """
    # Convert image to tensor
    image_tensor = T.ToTensor()(image).unsqueeze(0).to(device)

    # Make prediction
    model.eval()
    with torch.no_grad():
        prediction = model(image_tensor)

    # Get boxes, scores, and labels
    boxes = prediction[0]["boxes"]
    scores = prediction[0]["scores"]
    labels = prediction[0]["labels"]

    # Filter by confidence threshold
    valid_indices = scores > threshold
    boxes = boxes[valid_indices]
    scores = scores[valid_indices]
    labels = labels[valid_indices]

    # Enforce exactly one detection per class
    final_boxes = []
    final_labels = []

    for cls in range(1, num_classes + 1):  # Skip background (class 0)
        cls_indices = labels == cls
        if cls_indices.sum() > 0:
            cls_boxes = boxes[cls_indices]
            cls_scores = scores[cls_indices]
            # Select the box with the highest confidence
            best_idx = torch.argmax(cls_scores)
            final_boxes.append(cls_boxes[best_idx].tolist())
            final_labels.append(cls)

    return final_boxes, final_labels


def convert_result_to_pascal_voc(boxes, labels, class_names, image_size, output_path, image_name,folder_name):
    """
    Converts predicted boxes and labels to Pascal VOC XML format.

    Args:
        boxes (list or numpy.ndarray): List of bounding boxes in the format [xmin, ymin, xmax, ymax].
        labels (list or numpy.ndarray): List of corresponding class labels for the boxes.
        class_names (list): List of class names where index corresponds to the label.
        image_size (tuple): Image size as (width, height, depth).
        output_path (str): Path to save the Pascal VOC XML file.
        image_name (str): Name of the image (without the extension).

    Returns:
        None
    """
    # Create the root XML element
    annotation = ET.Element("annotation")

    # Add image metadata
    folder = ET.SubElement(annotation, "folder")
    folder.text = folder_name

    filename = ET.SubElement(annotation, "filename")
    filename.text = f"{image_name}"

    size = ET.SubElement(annotation, "size")
    width = ET.SubElement(size, "width")
    width.text = str(image_size[0])
    height = ET.SubElement(size, "height")
    height.text = str(image_size[1])
    depth = ET.SubElement(size, "depth")
    depth.text = "1"

    # Add each object
    for box, label in zip(boxes, labels):
        obj = ET.SubElement(annotation, "object")

        name = ET.SubElement(obj, "name")
        name.text = class_names[label]

        pose = ET.SubElement(obj, "pose")
        pose.text = "Unspecified"

        truncated = ET.SubElement(obj, "truncated")
        truncated.text = "0"

        difficult = ET.SubElement(obj, "difficult")
        difficult.text = "0"

        bndbox = ET.SubElement(obj, "bndbox")
        xmin = ET.SubElement(bndbox, "xmin")
        xmin.text = str(int(box[0]))
        ymin = ET.SubElement(bndbox, "ymin")
        ymin.text = str(int(box[1]))
        xmax = ET.SubElement(bndbox, "xmax")
        xmax.text = str(int(box[2]))
        ymax = ET.SubElement(bndbox, "ymax")
        ymax.text = str(int(box[3]))

    # Save the XML to a file
    tree = ET.ElementTree(annotation)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)


aut_label_dir = Path(r"/Users/igor.varha/CCP_workdir/full_body_workdir/automated_label")

# iterate over images

for image_file in os.listdir(folder_with_images):
    if (folder_with_labels/Path(image_file).with_suffix(".xml")).exists():
        print(f"Skipping {image_file}", (folder_with_labels/Path(image_file).with_suffix(".xml")))
        continue
    image = Image.open(folder_with_images/image_file).convert("RGB")
    boxes, labels = predict_image(model, image, device, threshold=0.2, num_classes=len(classes))
    convert_result_to_pascal_voc(boxes, labels, classes, image.size, aut_label_dir/Path(image_file).with_suffix(".xml"), image_file, aut_label_dir.name)


NameError: name 'classes' is not defined

In [41]:
# save model

torch.save(model.state_dict(), "/Users/igor.varha/CCP_workdir/full_body_workdir/top_model.pth")

In [38]:
list(range( len(classes)))

[0, 1, 2, 3, 4, 5]

In [26]:
dataset[0]

(tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],
 
         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]),
 {'boxes': tensor([[0., 0., 1., 1.]]),
  'labels': tensor([1]),
  'control_points': tensor([[414.5000, 254.5000],
          [414.5000,  88.5000]])})